In [1]:
from threading import Semaphore

from transformers import AutoModelForCausalLM, AutoTokenizer
from IPython.display import display, Markdown
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim

from ArxivSearch import ArxivSearch



In [2]:
def load_model():
    model_name = "Qwen/Qwen3-8B"
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype="auto",
        device_map="auto",
        local_files_only=True #swith it off to download from hugging face if you don't have it locally
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name, local_files_only=True)
    return {"model": model, "tokenizer": tokenizer}


In [3]:
def generate_arxive_query(model, tokenizer, prompt):
    system_prompt = f"Generate a search query for arxive extracting the key words out of this question: {prompt}.  Return the search query only"
    messages = [
        {"role": "user", "content": system_prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    print("Input tokens:", model_inputs["input_ids"].shape[1])
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens= 50
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
    answer = tokenizer.decode(
        output_ids,
        skip_special_tokens=True
    )

    return answer


In [4]:
model_and_tokenizer = load_model()
model = model_and_tokenizer["model"]
tokenizer = model_and_tokenizer["tokenizer"]

Loading weights:   0%|          | 0/399 [00:01<?, ?it/s]

In [5]:
prompt = "How to evaluate RAG system performance."
answer = generate_arxive_query(model, tokenizer, prompt)

Input tokens: 43


In [6]:
display(Markdown(answer))

"evaluating RAG system performance"

In [7]:
arxiv = ArxivSearch()

titles_to_urls = arxiv.search_titles("RAG evaluation", limit=5)

titles_to_urls



{'Mem++: Non-Destructive Memory for Long-Term Organizational LLM Agents': 'https://arxiv.org/pdf/2610.02002',
 'Mapping the RAG Landscape: A Four Axis Taxonomy of Efficiency, Defense, Interactivity, and Reasoning': 'https://arxiv.org/pdf/2610.01936',
 'A Matryoshka Hierarchical RAG for Efficient Multi-Hop Question Answering': 'https://arxiv.org/pdf/2610.01767',
 'AGO AI Quality Gate: Evidence-First Release Decisions for Retrieval-Augmented Generation': 'https://arxiv.org/pdf/2610.01218',
 'LawCompass: Navigating from Legal QA to Multi-Agent Deep Research with Grounded Evidence': 'https://arxiv.org/pdf/2610.01027'}

In [8]:
titles = list(titles_to_urls.keys())
sem_search = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = sem_search.encode(titles)
print(embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(5, 384)


In [9]:
from sentence_transformers import CrossEncoder
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [10]:
ranks = cross_encoder.rank(prompt, titles)
for rank in ranks:
    print(f"{rank['score']:.2f}\t{titles[rank['corpus_id']]}")

-3.31	Mapping the RAG Landscape: A Four Axis Taxonomy of Efficiency, Defense, Interactivity, and Reasoning
-5.71	A Matryoshka Hierarchical RAG for Efficient Multi-Hop Question Answering
-11.39	LawCompass: Navigating from Legal QA to Multi-Agent Deep Research with Grounded Evidence
-11.43	AGO AI Quality Gate: Evidence-First Release Decisions for Retrieval-Augmented Generation
-11.46	Mem++: Non-Destructive Memory for Long-Term Organizational LLM Agents


In [11]:
best_title = titles[ranks[0]['corpus_id']]
url = titles_to_urls[best_title]
article = arxiv.get_article_text(url)

In [13]:
article

'1\nMapping the RAG Landscape: A Four-Axis Taxonomy\nof Efficiency, Defense, Interactivity, and Reasoning\nMeghana Sunil1†, Shravya V1†, Shravan Venkatraman2, Joe Dhanith P R1∗\nAbstract—Large Language Models (LLMs) have demonstrated remarkable fluency across many tasks but remain limited by their\nstatic, parameter-bound knowledge and their susceptibility to hallucinating information. Retrieval-Augmented Generation (RAG)\naddresses these issues by incorporating external retrieval into the generation process, grounding model outputs in verifiable and\nup-to-date sources. While prior surveys primarily focus on core RAG architectures and standard pipelines, recent research explores\nbroader challenges and capabilities that extend beyond these foundational designs. This survey provides a consolidated and structured\nexamination of contemporary RAG developments, organizing the field into a four-axis taxonomy: improving retrieval efficiency,\nstrengthening robustness and security, supportin